# CoordAnalyst - an educational tool to analyse coordination complexes


Claire Bachmann, Bertille Delloye, Enéa Drezet--Marçot, Victor Davril


## Introduction

Coordination chemistry is an active field of research, with its study sitting at the intersection of many different branches of science, and its impact being major. In medicine, coordination compounds are indispensable. Cisplatin, one of the most widely used anticancer drugs in the world, is a simple platinum complex discovered in the 1960s that remains a frontline treatment for testicular, ovarian, and lung cancers. Another exmaple is Hemoglobin, the protein that carries oxygen through our blood, an iron coordination complex. In catalysis and industry, coordination chemistry drives some of the most economically significant chemical processes on the planet.The production of ammonia, acetaldehyde or even plastics, using the Haber-Bosch process, the Wacker process, and Ziegler-Natta catalysts, relies on iron, palladium and aluminium catalysts. In materials science, coordination compounds called metal-organic frameworks (MOFs) have emerged as one of the most exciting research areas of the 21st century, with applications in gas storage, carbon capture, drug delivery, and chemical sensing. Their extraordinary surface areas and tunable pore geometries make them uniquely powerful materials. In biology, nearly a third of all known proteins contain a metal ion at their active site : these coordination complexes perform catalysis, electron transfer, or structural stabilisation. Understanding these metalloenzymes is central to drug design and our understanding of life itself.  
Despite this enormous importance, coordination compounds remain challenging to characterise. Infrared and Raman spectroscopy are among the primary experimental tools used to study them, in order to reveal coordination modes, identify ligands, confirm geometries, and monitor reactions in real time. Yet interpreting these spectra requires deep chemical knowledge and access to reference data that is scattered across literature.  
Coordination chemistry is taught as a Bachelor's course in chemistry as the concepts are extremely relevant, and sometimes intertwine with spectroscopy and organometallic chemistry. Students are expected to aquire an understanding of metal and ligands, oxidation states, geometry, d-electrons, and bond stretches.    
Our package provides an educational tool with which students can try out different complexes' names and formulas and verify their properties, as well as visualize them in two and three dimensions. Moreover, CoordAnalyst provides an approximate IR and Raman spectra of the complex based on reference data from Nakamoto tables. It is presented as a unified interface, designed to make the spectral characterisation of coordination compounds more accessible to students and academics.



## Material and methods

### 1.1 Software architecture

The project is organised as a Python package, with the main source code stored in `src/coordchem`. The chemical data used by the program is kept outside the main source code, which separates the package functions from the reference data. Inside `coordchem`, the core modules are responsible for interpreting coordination complexes from user input. The file `parser.py` defines `parse_formula()`, which extracts the metal, ligands, charge, counter-ions, oxidation state and coordination number from a formula. The file `name.py` defines `parse_name()`, which performs a similar task starting from a coordination compound name. Both functions return the same structured object, `ParsedComplex`.

The file `complex.py` provides a higher-level interface through the `Complex` class. This class uses the formula and name parsers through methods such as `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()`. It gives direct access to the main properties of the complex and connects the parsed information to the geometry, visualisation and spectra modules.

The `geometry.py` module adds the chemical interpretation step. It contains functions such as `get_d_count()`, `get_geometry()`, `predict_geometry()` and `geometry_report()`. These functions use the parsed information to calculate the metal d-electron count and predict an idealised coordination geometry from the coordination number, metal, oxidation state and ligand information.

Two subpackages were created inside `coordchem`: `viz` and `spectra`. The `viz` subpackage contains the tools used for molecular visualisation. For 2D drawings, `ligand_data.py` stores ligand SMILES strings, donor atom indices and display labels. The file `diagram_2d.py` contains the main drawing functions, including `parse_complex_input()`, `build_coordination_mol()`, `diagram_2d_svg()` and `save_diagram_2d()`. The file `layout_2d.py` defines the idealised coordination-site positions with functions such as `coordination_sites()`, `chelate_octahedral_sites()`, `edta_octahedral_sites()` and `mixed_polydentate_site_groups()`. The file `transform_2d.py` contains the mathematical transformations used to position ligands, especially polydentate ligands, with functions such as `transform_monodentate()`, `transform_polydentate()`, `transform_acac()`, `transform_oxalate()` and `transform_edta()`.

The 3D visualisation is handled in `structure_3d.py`. This file defines idealised 3D coordination-site positions for different geometries, such as linear, tetrahedral, square planar and octahedral arrangements. These positions are selected with `geometry_positions()` and used by `build_complex_3d()` to assemble the metal and ligands into an RDKit molecule with a 3D conformer. The resulting structure can then be displayed interactively with `view_complex_3d()` or exported as HTML with `complex_3d_html()`.

The `spectra` subpackage contains the tools used for IR and Raman prediction. The `predictor.py` file extracts relevant spectral bands from the database for an input `ParsedComplex` and returns them as a `PredictionResult` object, ordered by increasing wavenumber. The `renderer.py` file then converts these bands into a simulated spectrum by representing each band as a Gaussian peak and summing the peaks to produce the final plotted spectrum.

### 1.2 Formula and Name Parser, Properties Extraction

The first step of the program is to transform the user input into a structured object that can be reused by the rest of the package. A coordination complex can be written as a formula, such as `[Fe(CN)6]4-`, or as a name, such as `tetraamminecopper(II)`. Although these two inputs look different, both are converted into the same internal format: a `ParsedComplex` object. This object stores the main chemical information of the complex, including the metal centre, ligands, ligand charges, denticity, donor atoms, complex charge, counter-ions, oxidation state and coordination number.

For formulas, this work is handled by `parser.py`, mainly through the function `parse_formula(formula: str) -> ParsedComplex`. This function reads the formula step by step. It separates possible counter-ions from the coordination sphere, identifies the metal centre, detects and counts the ligands, and extracts the global charge of the complex. The parser then enriches the result using internal dictionaries of known ligands, counter-ions and metals. From this information, it calculates derived properties such as the total ligand charge, the oxidation state of the metal and the coordination number.

For names, the same idea is implemented in `name.py` with the function `parse_name(name: str) -> ParsedComplex`. Instead of reading brackets and charge symbols, this function searches the compound name for known ligand names, metal names, prefixes and oxidation states written in Roman numerals. For example, in `tetraamminecopper(II)`, the program recognises `ammine` as the ligand, `copper` as the metal and `(II)` as the oxidation state. The name parser then returns the same `ParsedComplex` format as the formula parser. This parser is not intended to understand every possible IUPAC name, but it works for the common coordination names covered by the ligand and metal tables.

To make the package easier to use, the parsed information is wrapped in a higher-level `Complex` object defined in `complex.py`. This class acts as the main interface of the package. The methods `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()` allow the user to create the same type of object from different input formats. The `Complex` object gives direct access to useful properties such as the metal, ligands, oxidation state, coordination number, predicted geometry and d-electron count.

Once the complex has been parsed, the information can be passed to other modules. In particular, `geometry.py` uses functions such as `get_geometry()` and `get_d_count()` to predict the idealised geometry of the complex and calculate the number of d-electrons on the metal. This common structure is important because it allows the geometry, visualisation and spectra modules to work from the same chemical representation instead of handling formulas and names separately.

Overall, this part of the package acts as the bridge between human chemical notation and the rest of the code. It converts formulas and names into a clean chemical object, enriches it with useful properties, and prepares it for geometry prediction, visualisation and spectral analysis.

### 1.3 Spectral Band Database
The data was extracted systematically from  Nakamoto, K. "Infrared and Raman Spectra of Inorganic and Coordination Compounds", 6th ed., Wiley, 2009, and placed in SEED_BANDS : list of tuples with (ligand, coordination, metal, spectrum_type, wn_min, wn_max, intensity, assignment, ir_active, raman_active, source), which is defined at the top of the file as a BandRecord object. The data is given grouped by ligand for easy verification, and for both IR and Raman stretches.

Then the class IRBandDB is defined :  
The __init__ method does three things in order every time:  
-Opens a connection to SQLite  
-Creates the ir_bands table if it doesn't exist yet (_create_table)  
-Seeds the table with data if it's empty (_seed)  
   
First, _create_table runs a CREATE TABLE IF NOT EXISTS SQL statement. It defines the columns: ligand, coordination mode, metal, spectrum type, wavenumber range, intensity, assignment, whether it's IR/Raman active, and the source reference. Also creates two indexes to make lookups faster.
Then, _seed inserts all the rows from the SEED_BANDS list at the top of the file. It uses executemany which inserts all rows in one operation rather than one by one. The row count is checked first so it never double-inserts.  

The main query method is get_bands.   
(a) builds a SQL query with WHERE ligand = ? and AND spectrum_type = ?. The ? placeholders are essential to they prevent SQL injection and handle special characters safely.  
(b) if a coordination filter like "terminal" is passed, it adds AND coordination = ? to the query _row_to_record which return a list of BandRecord.  
(c) if a specific metal is passed : It splits the results into two groups: rows specific to that metal (e.g. metal = "Fe") and generic rows (metal = "any"). It then keeps all the specific rows, and only adds generic rows whose assignment isn't already covered by a specific row. This is how, for example, the Fe-specific 2093 cm⁻¹ band takes priority over the generic 2100–2200 range for the same C≡N stretch.  
get_all_ligands — returns a simple list of all ligand symbols in the database, e.g. ["CN", "CO", "Cl", "NH3", ...]. Useful for checking what's covered.  
get_bands_in_range(wmin, wmax) is used for reverse lookup : allows to identify an unknown peak in an experimental spectrum.  
add_band(...) is for inserting a custom entry. Useful when you encounter a ligand not in the seed data. Takes all the same fields as the database columns and commits immediately.  
summary(ligand) loops over IR then Raman bands and prints them as a formatted table, used for debugging and demos.  

### 1.4 Spectrum Prediction - Gaussians and plotting
**predictor.py** has a class PredictionResult which contains the attributes bands, intensities, the type of spectrum, the metal, the complex formula, ligand_coverage and warnings.  
The main function is predict_spectrum which returns a PredictionResult object from a ParsedComplex object. It loops over very ligand in the complex using te query db.get_bands for the specific ligand, spectrum and metal. The band intensity is scaled and not yet normalised according to the number of ligand of the same type. The bands extracted are sorted by order of increasing wavenumber.   
   
Chemical corrections : 
- Backbonding shifts for π-accepting ligands (CN, CO, NO) : These ligands accept electron density from the metal via π-backbonding.
More electron-rich metals (low oxidation state) → more backbonding → weaker C≡N / C≡O bond → LOWER wavenumber (negative shift)
Less electron-rich metals (high oxidation state) → less backbonding → stronger bond → HIGHER wavenumber (positive shift)
- Coordination shifts — how much a band moves when a free ligand coordinates. Negative = redshift (moves to lower wavenumber).
- Selection rules — which M-L band types are IR/Raman active per geometry.  
Based on the mutual exclusion rule: In centrosymmetric complexes (Oh, D4h), a band cannot be both IR and Raman active.
+++

**renderer.py** first defines the mathematical gaussian for variables x, center, height and sigma. In build_spectrum : first we use linspace for x to create an array of evenly spaced numbers between wn_range[0] and wn_range[1]. We have decided 3000 points as the more points we have, the smoother the curve appears. For y, originally we create an array of the same size containing only zero. Then, looping over every band, a gaussian peak is added to the right y position.  
plot_spectrum uses x, y from build_spectrum and traces the figure. A vertical dotted line is also added from the top of the peak to the position on the x-axis to be able to easily read off the wavenumber of the stretch. 

### 1.4 Two-Dimensional Visualisation

The two-dimensional visualisation module generates schematic drawings of coordination complexes. After the complex has been parsed and its geometry has been predicted, the program uses this information to place the metal centre at the centre of the drawing and arrange the ligands around it according to an idealised coordination geometry. The aim is not to reproduce an exact crystallographic structure, but to provide a clear and chemically meaningful representation of the coordination sphere.

The 2D drawing workflow is mainly handled in `diagram_2d.py`. The function `parse_complex_input()` allows the drawing functions to accept a formula, a compound name or an already parsed `ParsedComplex` object. The predicted geometry is then obtained with `get_geometry()` and used to decide how the ligands should be arranged around the metal.

The positions of the coordination sites are generated in `layout_2d.py` by the function `coordination_sites(geometry, n_sites)`. Depending on the predicted geometry, this function returns predefined positions around the metal centre. Linear complexes are drawn with two opposite sites, square planar complexes as a flat cross, tetrahedral complexes with wedge and dashed bonds to suggest depth, and octahedral complexes with a simplified projection containing two axial bonds and four surrounding bonds. For coordination number 8, the module can also generate a square-antiprismatic-type layout.

The drawing also depends on ligand-specific information stored in `ligand_data.py`. This file contains the SMILES strings used to describe each ligand, donor atom indices, and display labels. A SMILES string is a compact text representation of molecular connectivity. For example, ethylenediamine (`en`) is represented as `NCCN`, oxalate (`ox`) as `[O-]C(=O)C(=O)[O-]`, and acetylacetonate (`acac`) as `CC(=O)C=C(O)C`. These SMILES strings are converted by RDKit into `Chem.Mol` objects, which are connectivity-based molecular representations that RDKit can use to generate 2D coordinates and draw the ligand.

Small monodentate ligands, such as NH3, H2O, CN or Cl, can be drawn as compact labels to keep the diagram readable. Larger or polydentate ligands, such as ethylenediamine, oxalate, acetylacetonate or EDTA, are drawn using their RDKit-generated ligand structures. This allows the program to show the ligand backbone when it is chemically useful, while keeping the overall diagram clear.

The layout is based on simple coordination chemistry rules. The coordination number defines an ideal reference geometry: linear for coordination number 2, trigonal planar for coordination number 3, tetrahedral or square planar for coordination number 4, octahedral for coordination number 6, and square antiprismatic or dodecahedral for coordination number 8. However, real coordination complexes are rarely perfectly ideal. Their structures can depend on metal size, d-electron count, ligand steric effects and electronic interactions. For example, square planar geometries are often favoured for low-spin d⁸ metal centres such as Pt(II), Pd(II) or Ni(II) with strong-field ligands, while octahedral complexes may show distortions such as tetragonal elongation, tetragonal compression or Jahn--Teller distortion.

In our implementation, we chose a simplified idealised model rather than trying to reproduce these distortions quantitatively. The main objective was to avoid overlap between ligands and to keep the diagram easy to interpret. For this reason, specific layout functions were added for cases that are difficult to draw with a generic geometry. For example, `chelate_octahedral_sites()` is used for tris-bidentate octahedral complexes, `tridentate_octahedral_sites()` for bis-tridentate complexes, and `edta_octahedral_sites()` for EDTA-like hexadentate coordination.

A more general function, `mixed_polydentate_site_groups()`, was also added for complexes containing both polydentate and monodentate ligands. In this case, the polydentate ligands are assigned neighbouring coordination sites first, because they need several adjacent positions to bind correctly. The remaining sites are then filled by the monodentate ligands. This improves the readability of mixed-ligand complexes and reduces overlap problems.

After the coordination sites are chosen, the ligand structures are placed onto these sites. This is done in `build_coordination_mol()`, which creates a composite RDKit molecule containing the metal and all ligands. The function `_expand_ligands()` expands the ligand dictionary into a complete ligand list. Then `_make_ligand_mol()` takes the SMILES string from `ligand_data.py` and converts it into an RDKit `Chem.Mol` object with 2D coordinates. The function `_match_donor_indices()` identifies which atom or atoms of the ligand bind to the metal.

The ligand is then positioned using the transformation functions from `transform_2d.py`. Monodentate ligands are placed with `transform_monodentate()`, while polydentate ligands are placed with `transform_polydentate()`. Some ligands required more specific treatment to make the final drawing clearer, so special functions were added: `transform_acac()` for acetylacetonate, `transform_oxalate()` for oxalate, and `transform_edta()` for EDTA. These functions do not calculate real molecular conformations; they adjust the ligand orientation, scale and position so that the final diagram remains readable and chemically meaningful.

The final drawing is generated by `diagram_2d_svg()`. This function builds the full coordination molecule, draws it with RDKit, adds a title and returns the result as an SVG image. The SVG format was chosen because it gives clean and scalable figures that can easily be included in reports or presentations. The function `save_diagram_2d()` can then be used to save the SVG file.

Bond styles are used to make the drawings easier to interpret. Normal lines represent bonds in the drawing plane, while bold wedges and dashed bonds suggest approximate three-dimensional orientation. This is especially useful for tetrahedral and octahedral drawings. However, this remains a simplified convention. The current 2D module does not explicitly distinguish stereochemical isomers such as cis/trans, mer/fac or Λ/Δ. It provides a general idealised representation of the coordination environment rather than a full stereochemical model.

Overall, the 2D visualisation module follows a clear workflow: the input is parsed with `parse_complex_input()`, the geometry is predicted with `get_geometry()`, ligand-specific information is taken from `ligand_data.py`, coordination sites are generated by `coordination_sites()` or special layout functions, ligands are positioned with the transformation functions, and the final SVG drawing is produced by `diagram_2d_svg()` or saved with `save_diagram_2d()`.

### 1.5 Three-Dimentional Visualisation

### 1.6 Web Application - Streamlit interface

victor

### 1.7 Testing and Validation

The package was validated using a pytest test suite. The tests were organised by module in order to check each part of the workflow separately before testing more integrated behaviours. This was important because the package combines several steps: parsing the input, extracting chemical properties, predicting the geometry, generating visualisations and predicting spectral bands.

The parser was tested in `test_parser.py`. These tests check that formulas such as `[Fe(CN)6]4-`, `[PtCl2(NH3)2]`, `[Co(en)3]3+` and `K4[Fe(CN)6]` are correctly interpreted. They verify the extraction of the metal, ligands, complex charge, counter-ions, ligand metadata, oxidation state and coordination number. Edge cases were also tested, such as unknown ligands, missing charge suffixes, whitespace in the input and invalid metal symbols.

The name parser was tested in `test_name.py`. These tests verify that names such as `tetraamminecopper(II)`, `hexacyanoferrate(II)` and `diamminedichloroplatinum(II)` are converted into the same type of `ParsedComplex` object as formula inputs. They also check the extraction of oxidation states from Roman numerals, the recognition of metal names and the parsing of ligand prefixes such as di-, tetra- and hexa-.

The `Complex` class was tested in `test_complex.py`. These tests check that `Complex.from_formula()`, `Complex.from_name()` and `Complex.from_input()` correctly build a high-level `Complex` object. They also verify that the class exposes the expected properties, including the metal, ligands, oxidation state, coordination number, predicted geometry and d-electron count.

The geometry module was tested in `test_geometry.py`. These tests validate the rule-based geometry prediction for common textbook examples, such as linear silver complexes, octahedral hexacyanoferrate, square-planar platinum(II) complexes and ambiguous four-coordinate nickel or copper complexes. The same test file also checks the calculation of the metal d-electron count.

The 2D visualisation module was tested in `test_diagram_2d.py`. These tests check that SVG diagrams are generated correctly, that files can be saved, and that different geometries produce the expected idealised projections. More specific tests validate the treatment of chelating ligands, EDTA, mixed polydentate/monodentate complexes, compact labels for small ligands, wedge/dash bond styles and square-antiprismatic projections.

The 3D visualisation module was tested in `test_structure_3d.py`. These tests check the generation of idealised coordination-site positions, the creation of 3D ligand structures from SMILES strings, donor atom detection and the construction of a complete RDKit molecule with a 3D conformer. They also verify that the `Complex` class can access the 3D building and HTML display functions.

Finally, the spectral prediction part was tested using `test_ir_ra_bands.py` and `test_predictor.py`. These tests verify that the spectral database contains the expected ligands and band records, that IR and Raman bands can be extracted, and that the predictor returns a `PredictionResult` object with bands sorted by increasing wavenumber. Benchmark complexes such as hexacyanoferrate, cisplatin and chromium hexacarbonyl were used to check that characteristic spectral regions are represented.

Overall, the tests show that the main workflow of the package is functional: a complex can be parsed from a formula or a name, converted into chemical properties, assigned a predicted geometry, visualised in 2D or 3D, and connected to spectral prediction.

## Results

validation of geometry ? visualisations issues

### Database coverage
 IR spectroscopy is widely used to characterize coordination compounds, especially for identifying ligands such as CO, CN⁻, and NO, and distinguishing binding modes (e.g., SCN vs NCS). However, accurate spectral prediction typically requires quantum chemical calculations, which are computationally intensive and not always accessible. There is therefore value in a lightweight, interpretable tool that can provide rapid, approximate spectral predictions and assist with spectral assignment. The focus is on interpretability and rapid qualitative analysis rather than high-precision prediction.
The scope is restricted to a selected set of common ligands, including CO, CN⁻, NO, SCN⁻/NCS⁻, NO₂⁻/ONO⁻, NH₃, H₂O, halides, and several chelating ligands. (CN, CO, NH3, H2O, Cl, Br, I, OH, NO2, ONO, SCN, NCS, NO, N3, en, ox, acac, py, dmso, PPh3. ) The tool focuses on characteristic group frequencies rather than full vibrational analysis. 


## Discussion


### 3.1 Accuracy and limitations

The current implementation provides a useful educational representation of coordination complexes, but it remains limited to relatively simple systems. The package is mainly designed for mononuclear complexes, meaning complexes containing one metal centre. It does not currently handle polynuclear complexes, metal clusters or complexes with bridging ligands. For example, ligands such as μ-Cl, μ-CN or bridging carbonyls are not explicitly represented as links between two metal centres. This is an important limitation, because many real coordination compounds contain more than one metal centre or bridging coordination modes.

The parser is also based on simplified chemical notation. Formula parsing works well for common coordination formulas, and name parsing works for names containing known ligand names, metal names, prefixes and oxidation states written as Roman numerals. However, the program does not yet support all possible IUPAC naming conventions. For example, it currently uses names such as `tetraamminecopper(II)`, where the Roman numeral gives the metal oxidation state. Some nomenclature systems can also indicate the charge of the coordination sphere directly, for example with numerical charge descriptors. In such cases, the oxidation state would need to be deduced from the total ligand charge and the charge of the complex. This more general name interpretation is not yet implemented.

The visualisation modules are also idealised. In 2D, the program draws only the coordination sphere and does not represent counter-ions around the complex. Small monodentate ligands, such as NH3, H2O, CN or Cl, are often shown using abbreviated labels to keep the drawing readable. This makes the figures clearer, but it also means that the complete molecular structure of these ligands is not always shown. Larger ligands are drawn from SMILES using RDKit, but the representation is still schematic and does not correspond to a crystallographic structure.

The 2D layout is based on idealised geometries and simplified projections. This works well for common geometries such as linear, square planar, tetrahedral and octahedral complexes. However, more complex geometries are harder to represent accurately in a flat drawing. For example, dodecahedral environment can appear flattened in 2D, and not all of them use wedge or dashed bonds because the depth information becomes difficult to represent clearly. Similarly, cyclic ligands are generally drawn flat, although in real complexes they may adopt conformations with more three-dimensional depth.

The 3D visualisation is also limited by the complexity of the ligands. The current module can generate idealised 3D structures for relatively simple ligands, but more complex polydentate ligands are harder to position correctly. For this reason, ligands such as dien or tren were not fully implemented, because their flexible multidentate coordination would require more advanced conformational placement. The current 3D structures should therefore be understood as approximate educational models rather than experimentally optimised geometries.

Finally, the spectral prediction is approximate. The IR and Raman spectra are generated from reference band ranges and simple Gaussian peaks, not from quantum chemical vibrational calculations. This makes the spectra fast and interpretable, but they should not be considered as high-precision predictions. The aim is to help students identify characteristic ligand vibrations and trends, not to replace experimental spectra or computational chemistry.

### 3.2 Value and comparison with existing tools

The main value of CoordAnalyst is that it brings several coordination chemistry tasks together in one educational workflow. Starting from a simple formula or name, the user can obtain the metal and ligands, oxidation state, coordination number, predicted geometry, d-electron count, 2D and 3D visualisations, and approximate IR or Raman spectra. This helps connect concepts that are often taught separately, such as nomenclature, ligand denticity, geometry and vibrational spectroscopy.

Compared with general cheminformatics tools such as RDKit, CoordAnalyst is more specialised for coordination complexes. RDKit is used for molecular representation and drawing, but it does not directly calculate coordination-specific properties such as ligand denticity, metal oxidation state or coordination geometry. CoordAnalyst adds these chemical rules on top of RDKit to make the results more useful for coordination chemistry.

The package is less accurate than molecular visualisation or quantum chemistry software, because it does not use experimental structures, geometry optimisation or ab initio vibrational calculations. However, this is also what makes it fast, transparent and accessible. It can generate idealised structures and approximate spectra directly from user input, without requiring a pre-existing structure file or advanced computational setup.

CoordAnalyst is therefore not intended to replace professional modelling or spectroscopy tools. Its value lies in providing a lightweight, interpretable and modular support tool for teaching and learning coordination chemistry. New ligands, spectral bands or visualisation rules can be added without rewriting the whole package.

### 3.3 Future Improvements

Several improvements could be made to extend the current package. One important improvement would be to handle stereochemical information more explicitly. In the current version, the 2D representation does not automatically generate specific cis/trans, mer/fac or Λ/Δ isomers when the input name or formula does not specify the exact configuration. This is a deliberate limitation, because the same formula can correspond to several stereoisomers, and choosing one randomly could be chemically misleading.

However, when the stereochemistry is clearly given in the input, the package could be improved to use this information. For example, the name parser could be extended to recognise descriptors such as cis, trans, fac, mer, Λ and Δ. The 2D visualisation module could then place the ligands in the correct relative positions and draw the requested stereoisomer explicitly. This would make the package more accurate while keeping the default behaviour safe: unspecified structures would remain idealised, while specified stereochemical names would produce the corresponding configuration. This would require not only changes in the 2D drawing module, but also an extension of the name parser so that stereochemical descriptors are stored in the parsed object and passed to the visualisation functions.

The name parser could also be extended to support additional nomenclature conventions. At the moment, the program mainly interprets oxidation states written as Roman numerals, such as iron(III) or copper(II). A future version could also recognise names where the charge of the coordination sphere is given directly. In that case, the program would deduce the oxidation state of the metal from the complex charge and the total ligand charge. This would make the parser more flexible and closer to real coordination nomenclature.

Another important improvement would be to extend the range of complexes that can be represented. The current package is mostly limited to mononuclear complexes with terminal ligands. Future versions could include polynuclear complexes, bridging ligands and metal clusters. This would require a more complex internal representation, because a ligand would need to be connected to more than one metal centre. It would also require new drawing rules for bridging coordination modes such as μ-Cl, μ-CN or bridging carbonyl ligands.

The visualisation modules could also be improved. In 2D, more realistic depth information could be added for cyclic ligands and complex geometries. At the moment, cyclic ligands are mostly drawn flat, and geometries such as dodecahedral or square-antiprismatic coordination are simplified in the projection. Future versions could use more wedge and dashed bonds to represent depth more clearly, although this would need to be balanced against readability. More realistic distortions could also be added, such as Jahn--Teller elongation in Cu(II) complexes, tetragonal compression or trigonal distortion.

The 3D visualisation could be extended to support more complex polydentate ligands. Ligands such as dien or tren were not included because their flexible multidentate coordination is difficult to place automatically in a simple idealised model. A future version could use more advanced conformer generation and ligand-placement algorithms to position these ligands more realistically around the metal centre. Approximate metal--ligand bond distances could also be included using experimental data or a curated database.

Another possible improvement would be to let the user draw the molecule directly in the interface. This could be done using a chemical structure editor such as Ketcher or JSME integrated into the web application. The program could then extract the molecular information from the drawing, identify the metal centre, ligands and donor atoms, and convert the drawing into the same internal representation used by the rest of the package. This would make the tool more interactive and would reduce the need for the user to know the exact formula or name syntax.

Finally, the export and reporting options could be improved. The module currently generates SVG drawings, which are useful because they are clean and scalable. Direct export to PNG or PDF, together with more control over figure size, labels, bond style and counter-ion display, would make the visualisation module easier to use in reports, presentations and teaching material. The spectral module could also be improved by allowing users to compare predicted spectra with uploaded experimental spectra.

# Conclusion